# 10a -- MTMC Stages 0-2: Frame Extraction + Detection + 3-Model ReID Features

**Run once (or when tracking/ReID config changes). ~110-130 min on P100.**

| Stage | What | Time |
|---|---|---|
| 0 | Frame extraction (10 fps, 6 cameras) | ~20 min |
| 1 | YOLO detection + BotSort tracking | ~45 min |
| 2 | TransReID 768D + ResNet101-IBN 2048D + ResNeXt101-IBN 2048D -> PCA 384D features | ~30 min |

After this runs, its output (`checkpoint.tar.gz`) is used by **10b** -> **10c**.
Attach `mtmc-weights` via **Add Data -> Your Datasets -> mtmc-weights**.

In [ ]:
import os, sys, subprocess, shutil, json, time, tarfile, re
from pathlib import Path

# --- Guard: detect GPU BEFORE importing torch ---
# Kaggle's PyTorch 2.10+ drops P100 (sm_60) support.
# If we got a P100, downgrade to a compatible build first.
if shutil.which("nvidia-smi"):
    _nvsmi = subprocess.run(
        ["nvidia-smi", "--query-gpu=gpu_name,compute_cap", "--format=csv,noheader"],
        capture_output=True, text=True)
    if _nvsmi.returncode == 0 and _nvsmi.stdout.strip():
        _gpu_name, _cap = _nvsmi.stdout.strip().split(",", 1)
        _match = re.search(r"(\d+)\.(\d+)", _cap)
        if _match:
            _major, _minor = _match.groups()
            _sm = int(_major) * 10 + int(_minor)
            if _sm < 70:
                print(f"\u26a0 GPU {_gpu_name.strip()} (sm_{_sm}) — installing compatible PyTorch ...")
                subprocess.check_call([
                    sys.executable, "-m", "pip", "install", "-q",
                    "torch==2.4.1", "torchvision==0.19.1",
                    "--index-url", "https://download.pytorch.org/whl/cu124",
                ])
                print("\u2713 Compatible PyTorch installed")

import torch

print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA   : {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  ({p.total_memory/1024**3:.1f} GB)")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsing device: {DEVICE}")

## 1. Clone Repo & Install Dependencies

In [ ]:
REPO_URL = "https://github.com/MRKDaGods/gp.git"
WORK_DIR = Path("/kaggle/working")
PROJECT  = WORK_DIR / "gp"

if not PROJECT.exists():
    print(f"Cloning {REPO_URL} ...")
    subprocess.check_call(["git", "clone", "--depth", "1", "-b", "feature/people-tracking", REPO_URL, str(PROJECT)])
else:
    print("Repo already present, pulling latest ...")
    subprocess.check_call(["git", "-C", str(PROJECT), "pull", "--ff-only"])

os.chdir(str(PROJECT))
sys.path.insert(0, str(PROJECT))
print(f"\n\u2713 Repo ready at {PROJECT}")

In [ ]:
    import subprocess, sys

    def pip(*args):
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

    # Keep Kaggle's preinstalled torch/torchvision/numpy stack intact.
    pip("--no-deps", "ultralytics")
    pip("filterpy", "ftfy", "lapx")
    pip("--no-deps", "boxmot==11.0.3")

    try:
        import torchreid; print("torchreid ok")
    except ImportError:
        pip("git+https://github.com/KaiyangZhou/deep-person-reid.git")

    try:
        import faiss; print(f"faiss ok ({faiss.__version__})")
    except ImportError:
        try: pip("faiss-gpu")
        except Exception: pip("faiss-cpu")

    pip("timm", "motmetrics")
    pip("open_clip_torch")
    pip("loguru", "omegaconf", "rich", "networkx>=3.1", "click")

    # SAM2 - non-torch deps first, then SAM2 itself without deps
    pip("hydra-core", "iopath")
    pip("--no-deps", "git+https://github.com/facebookresearch/sam2.git")

    # Install project in editable mode (no deps - all handled above)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"])
    print("
✓ All dependencies installed")

In [ ]:
    FAILED = []
    _checks = [
        ("ultralytics", "ultralytics"),
        ("boxmot", "boxmot"),
        ("torch", "torch"),
        ("torchreid", "torchreid"),
        ("timm", "timm"),
        ("open_clip", "open_clip"),
        ("faiss", "faiss"),
        ("motmetrics", "motmetrics"),
        ("cv2", "cv2"),
        ("loguru", "loguru"),
        ("omegaconf", "omegaconf"),
        ("networkx", "networkx"),
        ("sklearn", "sklearn"),
        ("numpy", "numpy"),
        ("pandas", "pandas"),
        ("sam2", "sam2"),
    ]
    for label, mod in _checks:
        try:
            __import__(mod)
            print(f"  ✓ {label}")
        except ImportError as e:
            print(f"  ✗ {label}: {e}")
            FAILED.append(label)
    if FAILED:
        raise RuntimeError(f"Missing modules: {FAILED} -- fix pip installs above")
    print("
✓ All required modules importable")

## 2. Mount Model Weights
Model weights dataset (`mtmc-weights`) must be attached.

In [ ]:
    WEIGHTS_INPUT = Path("/kaggle/input/mtmc-weights")
    assert WEIGHTS_INPUT.exists(), (
        "Dataset 'mtmc-weights' not found.
"
        f"  Expected: {WEIGHTS_INPUT}
"
        "  Attach via: Add Data -> Your Datasets -> mtmc-weights"
    )

    MODELS_DST = PROJECT / "models"
    if MODELS_DST.is_symlink(): MODELS_DST.unlink()
    if MODELS_DST.exists(): shutil.rmtree(MODELS_DST)
    print(f"Copying models/ from {WEIGHTS_INPUT} (~750 MB) ...")
    shutil.copytree(str(WEIGHTS_INPUT), str(MODELS_DST))

    # --- Debug: show what was copied ---
    print("Contents of models/ after copy:")
    for f in sorted(MODELS_DST.rglob("*"))[:30]:
        if f.is_file():
            print(f"  {f.relative_to(MODELS_DST)} ({f.stat().st_size/1024**2:.1f} MB)")

    # --- Extract any zip files (--dir-mode zip uploads) ---
    import zipfile
    for zf in sorted(MODELS_DST.rglob("*.zip")):
        print(f"Extracting {zf.relative_to(MODELS_DST)} ...")
        with zipfile.ZipFile(zf) as z:
            z.extractall(zf.parent)
        zf.unlink()

    # --- Handle flat structure: move root-level model files into subdirs ---
    for subdir in ["detection", "reid", "tracker"]:
        (MODELS_DST / subdir).mkdir(exist_ok=True)

    for f in list(MODELS_DST.glob("*.pt")):
        if "yolo" in f.name.lower():
            shutil.move(str(f), str(MODELS_DST / "detection" / f.name))
        elif "osnet" in f.name.lower():
            shutil.move(str(f), str(MODELS_DST / "tracker" / f.name))
    for f in list(MODELS_DST.glob("*.pth")):
        shutil.move(str(f), str(MODELS_DST / "reid" / f.name))
    for f in list(MODELS_DST.glob("*.pkl")):
        shutil.move(str(f), str(MODELS_DST / "reid" / f.name))
    for f in list(MODELS_DST.glob("*.json")):
        if f.name != "dataset-metadata.json":
            shutil.move(str(f), str(MODELS_DST / "reid" / f.name))

    print("
Final models/ structure:")
    for f in sorted(MODELS_DST.rglob("*")):
        if f.is_file():
            print(f"  {f.relative_to(MODELS_DST)} ({f.stat().st_size/1024**2:.1f} MB)")

    ESSENTIAL = [
        "models/detection/yolo26m.pt",
        "models/reid/transreid_cityflowv2_best.pth",
        "models/tracker/osnet_x0_25_msmt17.pt",
    ]
    missing = [p for p in ESSENTIAL if not (PROJECT / p).exists()]
    if missing:
        for m in missing: print(f"  MISSING: {m}")
        raise FileNotFoundError("Fix missing weights before continuing.")
    print("✓ All essential baseline weights present")

    # --- Copy CLIP RN50x4 CNN secondary model from 09m kernel output ---
    CNN_RN50X4_SRC = Path("/kaggle/input/09m-clip-rn50x4-vehicle-reid-cityflowv2/exported_models/clip_rn50x4_cityflowv2.pth")
    CNN_RN50X4_DST = PROJECT / "models" / "reid" / "clip_rn50x4_cityflowv2.pth"
    if CNN_RN50X4_SRC.exists():
        shutil.copy2(str(CNN_RN50X4_SRC), str(CNN_RN50X4_DST))
        print(f"✓ CLIP RN50x4 CNN: {CNN_RN50X4_DST.name} ({CNN_RN50X4_DST.stat().st_size/1024**2:.0f} MB)")
    else:
        print(f"⚠ CLIP RN50x4 weights not found at {CNN_RN50X4_SRC} -- CNN secondary extraction disabled")
        CNN_RN50X4_DST = None

## 2b. Vehicle ReID Model

Use the baseline CityFlowV2 TransReID checkpoint from `mtmc-weights`.
Foreground masking will be applied at inference time in Stage 2.

In [ ]:
# --- Baseline ReID weights used by this notebook ---
_primary_path = PROJECT / "models" / "reid" / "transreid_cityflowv2_best.pth"
print(f"✓ Primary 256px ViT: {_primary_path.name} ({_primary_path.stat().st_size/1024**2:.0f} MB)")
if CNN_RN50X4_DST and CNN_RN50X4_DST.exists():
    print(f"✓ Secondary CNN ready: {CNN_RN50X4_DST.name}")
else:
    print("✓ Secondary CNN extraction: disabled")

## 3. Mount CityFlowV2 Dataset

CityFlowV2 is attached as a Kaggle dataset. We symlink the nested
`train/S01/c001/` structure into the flat `S01_c001/` layout the pipeline expects.

In [ ]:
import re as _re, shutil as _shutil

for mount in ["/tmp", "/kaggle/working"]:
    total, used, free = _shutil.disk_usage(mount)
    print(f"{mount:20s}  {free/1024**3:.1f} GB free / {total/1024**3:.1f} GB total")

# --- Locate the Kaggle-mounted CityFlowV2 dataset ---
_CANDIDATE_MOUNTS = [
    Path("/kaggle/input/data-aicity-2023-track-2"),
    Path("/kaggle/input/datasets/thanhnguyenle/data-aicity-2023-track-2"),
]
CITYFLOW_INPUT = None
for _p in _CANDIDATE_MOUNTS:
    if _p.exists():
        CITYFLOW_INPUT = _p
        break
assert CITYFLOW_INPUT is not None, (
    "CityFlowV2 dataset not found.\n"
    "  Attach via: Add Data -> Datasets -> search 'data-aicity-2023-track-2'"
)
print(f"\u2713 CityFlowV2 dataset at {CITYFLOW_INPUT}")

# --- Flatten into data/raw/cityflowv2/S01_c001/ layout ---
TMP_DATA = Path("/tmp/datasets")
TMP_DATA.mkdir(parents=True, exist_ok=True)

DATA_RAW_PARENT = PROJECT / "data" / "raw"
if not DATA_RAW_PARENT.is_symlink():
    if DATA_RAW_PARENT.exists(): shutil.rmtree(DATA_RAW_PARENT)
    DATA_RAW_PARENT.parent.mkdir(parents=True, exist_ok=True)
    DATA_RAW_PARENT.symlink_to(TMP_DATA)
    print(f"\u2713 data/raw \u2192 {TMP_DATA}")
else:
    print(f"data/raw already symlinked -> {DATA_RAW_PARENT.resolve()}")

DATA_OUT = Path("/tmp/pipeline_outputs")
DATA_OUT.mkdir(parents=True, exist_ok=True)
DATA_RAW = TMP_DATA / "cityflowv2"
DATA_RAW.mkdir(parents=True, exist_ok=True)

# Map: train/S01/c001 -> S01_c001, validation/S02/c006 -> S02_c006
for split_dir in sorted(CITYFLOW_INPUT.iterdir()):
    if not split_dir.is_dir() or split_dir.name not in ("train", "validation", "test"):
        continue
    for scene_dir in sorted(split_dir.iterdir()):
        if not scene_dir.is_dir():
            continue
        for cam_dir in sorted(scene_dir.iterdir()):
            if not cam_dir.is_dir():
                continue
            flat_name = f"{scene_dir.name}_{cam_dir.name}"  # e.g. S01_c001
            flat_dir = DATA_RAW / flat_name
            if flat_dir.exists():
                continue
            # Symlink the entire camera dir (read-only mount is fine for inference)
            flat_dir.symlink_to(cam_dir)

CAM_RE = _re.compile(r"^S\d{2}_c\d{3}$")
cams = sorted(d.name for d in DATA_RAW.iterdir() if d.is_dir() and CAM_RE.match(d.name))
print(f"\u2713 CityFlowV2 ready: {len(cams)} cameras")
for c in cams: print(f"  {c}")
print(f"\nDataset path: {DATA_RAW}")

## 3b. Generate ROI Masks

Generate per-camera ROI masks from video using background subtraction.
These masks eliminate non-road detections (buildings, parked cars) and are
the single biggest factor in reducing false positives.

In [ ]:
os.chdir(str(PROJECT))
print("Generating ROI masks ...")
r = subprocess.run(
    [sys.executable, "scripts/generate_roi_masks.py",
     "--data-dir", str(DATA_RAW),
     "--n-samples", "200"],
    cwd=str(PROJECT))
if r.returncode != 0:
    print("WARNING: ROI mask generation failed (non-fatal, will proceed without masks)")
else:
    # Verify masks were created
    masks = list(DATA_RAW.glob("*/roi.jpg"))
    print(f"\u2713 Generated {len(masks)} ROI masks")

## 4. Run Stages 0-2

This cell restores the exact v80 override set that produced MTMC IDF1=0.784.
Do not change the listed stage1/stage2 overrides without re-running downstream association sweeps.

In [ ]:
from datetime import datetime
RUN_NAME = f"run_kaggle_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
RUN_DIR  = DATA_OUT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
BENCHMARK_CAMERAS = ['S01_c001', 'S01_c002', 'S01_c003', 'S02_c006', 'S02_c007', 'S02_c008']
print(f"Run  : {RUN_NAME}")
print(f"Cams : {BENCHMARK_CAMERAS}")

In [ ]:
os.chdir(str(PROJECT))
import os
import subprocess, time, sys

BASELINE_WEIGHTS = "models/reid/transreid_cityflowv2_best.pth"
print(f"Baseline vehicle model: {BASELINE_WEIGHTS} (exists={os.path.exists(BASELINE_WEIGHTS)})")

cmd = [
    sys.executable, "scripts/run_pipeline.py",
    "--config", "configs/default.yaml",
    "--dataset-config", "configs/datasets/cityflowv2.yaml",
    "--stages", "0,1,2",
    "--override", f"project.run_name={RUN_NAME}",
    "--override", f"project.output_dir={DATA_OUT}",
    "--override", "stage0.cameras=[S01_c001,S01_c002,S01_c003,S02_c006,S02_c007,S02_c008]",
    # v80-restored Stage 0-2 baseline with SAM2 foreground masking disabled
    "--override", "stage2.crop.samples_per_tracklet=48",
    "--override", "stage2.crop.foreground_masking.enabled=false",
    "--override", "stage2.pca.n_components=384",
    "--override", "stage2.reid.color_augment=false",
    "--override", "stage2.reid.vehicle.concat_patch=true",
    "--override", "stage2.reid.vehicle.input_size=[256,256]",
    "--override", "stage2.multi_query.k=0",
    "--override", "stage2.camera_bn.enabled=false",
    "--override", "stage2.camera_tta.enabled=false",
    "--override", "stage2.power_norm.alpha=0.0",
    "--override", "stage1.interpolation.max_gap=50",
    "--override", "stage1.intra_merge.max_time_gap=40.0",
    "--override", "stage1.tracker.min_hits=2",
]

print("CMD:", " ".join(str(c) for c in cmd))
print("=" * 70)
t0 = time.time()
r = subprocess.run(cmd, cwd=str(PROJECT))
print("=" * 70)
elapsed = time.time() - t0
if r.returncode != 0:
    print(f"✗ FAILED after {elapsed/60:.1f} min"); sys.exit(r.returncode)
print(f"✓ Stages 0-2 done in {elapsed/60:.1f} min")

## 4b. CNN Secondary Feature Extraction (CLIP RN50x4)

Extract 768D CNN embeddings from the same tracklet crops using the CLIP RN50x4
model trained in 09m. These embeddings are PCA-whitened to 384D and saved as
`embeddings_secondary.npy` for score-level fusion in 10c.

The CNN uses the same `embedding_index.json` ordering as the primary stage2 output,
so row `i` in `embeddings_secondary.npy` always matches row `i` in `embeddings.npy`.

In [ ]:
    import json as _json
    import time as _cnn_time

    _cnn_t0 = _cnn_time.time()
    stage2_dir = RUN_DIR / "stage2"
    stage1_dir = RUN_DIR / "stage1"
    stage0_dir = RUN_DIR / "stage0"

    if CNN_RN50X4_DST is None or not CNN_RN50X4_DST.exists():
        print("⚠ CLIP RN50x4 weights not available -- skipping CNN secondary extraction")
    else:
        import cv2
        import numpy as np
        import open_clip
        import torch
        import torch.nn as nn
        import torch.nn.functional as F
        from omegaconf import OmegaConf

        from src.core.io_utils import load_tracklets_by_camera
        from src.stage2_features.crop_extractor import CropExtractor
        from src.stage2_features.embeddings import camera_aware_batch_normalize, l2_normalize
        from src.stage2_features.pca_whitening import PCAWhitener
        from src.stage2_features.pipeline import _load_frames_for_camera

        class CLIPResNetReID(nn.Module):
            """CLIP RN50x4 visual encoder + projection + BNNeck for ReID inference."""

            def __init__(self, backbone_dim=640, embed_dim=768):
                super().__init__()
                clip_model = open_clip.create_model("RN50x4", pretrained=None)
                self.backbone = clip_model.visual
                self.backbone_dim = backbone_dim
                self.embed_dim = embed_dim
                self.proj = nn.Linear(self.backbone_dim, self.embed_dim, bias=False)
                nn.init.kaiming_normal_(self.proj.weight, mode="fan_out")
                self.bn = nn.BatchNorm1d(self.embed_dim)
                self.bn.bias.requires_grad_(False)
                self.cls_head = nn.Linear(self.embed_dim, 1, bias=False)
                nn.init.normal_(self.cls_head.weight, std=0.001)
                del clip_model

            def forward(self, x):
                backbone_feat = self.backbone(x)
                feat = self.proj(backbone_feat)
                bn_feat = self.bn(feat)
                return F.normalize(bn_feat, p=2, dim=1)

        run_cfg_path = RUN_DIR / "config.yaml"
        if not run_cfg_path.exists():
            raise FileNotFoundError(f"Missing saved run config: {run_cfg_path}")

        cfg = OmegaConf.load(run_cfg_path)
        stage_cfg = cfg.stage2
        quality_temperature = float(stage_cfg.reid.get("quality_temperature", 3.0))
        flip_augment = bool(stage_cfg.reid.get("flip_augment", True))
        cnn_batch_size = 64
        target_dim = 384

        primary_embeddings = np.load(stage2_dir / "embeddings.npy")
        with open(stage2_dir / "embedding_index.json", "r", encoding="utf-8") as f:
            index_map = _json.load(f)
        if primary_embeddings.shape[0] != len(index_map):
            raise RuntimeError(
                f"Primary embeddings/index mismatch: {primary_embeddings.shape[0]} rows vs {len(index_map)} index entries"
            )
        print(f"Primary embeddings: {primary_embeddings.shape}")

        cnn_model = CLIPResNetReID(backbone_dim=640, embed_dim=768)
        ckpt = torch.load(str(CNN_RN50X4_DST), map_location="cpu", weights_only=False)
        state = ckpt.get("state_dict", ckpt)
        state = {k.replace("module.", "", 1) if k.startswith("module.") else k: v for k, v in state.items()}
        missing, unexpected = cnn_model.load_state_dict(state, strict=False)
        critical_missing = [k for k in missing if not k.startswith("cls_head")]
        if critical_missing:
            raise RuntimeError(f"CNN model missing critical keys: {critical_missing}")
        if unexpected:
            print(f"⚠ Unexpected checkpoint keys: {unexpected}")
        cnn_model.eval().to(DEVICE)
        if DEVICE == "cuda":
            cnn_model.half()
        print(f"✓ CLIP RN50x4 loaded: 768D embeddings, 288x288 inputs, device={DEVICE}")

        clip_mean = np.array([0.48145466, 0.4578275, 0.40821073], dtype=np.float32)
        clip_std = np.array([0.26862954, 0.26130258, 0.27577711], dtype=np.float32)
        cnn_h, cnn_w = 288, 288

        def preprocess_crops(batch_crops):
            processed = []
            for crop in batch_crops:
                rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
                rgb = cv2.resize(rgb, (cnn_w, cnn_h), interpolation=cv2.INTER_CUBIC)
                rgb = rgb.astype(np.float32) / 255.0
                rgb = (rgb - clip_mean) / clip_std
                processed.append(np.transpose(rgb, (2, 0, 1)))
            return torch.from_numpy(np.stack(processed, axis=0))

        @torch.no_grad()
        def extract_batch(batch_crops):
            tensor = preprocess_crops(batch_crops).to(DEVICE)
            if DEVICE == "cuda":
                tensor = tensor.half()
            features = cnn_model(tensor).float().cpu().numpy()
            views = 1
            if flip_augment:
                flipped = [cv2.flip(crop, 1) for crop in batch_crops]
                flip_tensor = preprocess_crops(flipped).to(DEVICE)
                if DEVICE == "cuda":
                    flip_tensor = flip_tensor.half()
                features = features + cnn_model(flip_tensor).float().cpu().numpy()
                views += 1
            return features / views

        def extract_crops_embeddings(batch_crops, batch_size=64):
            if not batch_crops:
                return np.empty((0, 768), dtype=np.float32)
            outputs = []
            for start in range(0, len(batch_crops), batch_size):
                batch = batch_crops[start:start + batch_size]
                try:
                    outputs.append(extract_batch(batch))
                except RuntimeError as exc:
                    if "out of memory" not in str(exc).lower():
                        raise
                    if not torch.cuda.is_available():
                        raise
                    torch.cuda.empty_cache()
                    sub_batch_size = max(1, len(batch) // 2)
                    for offset in range(0, len(batch), sub_batch_size):
                        outputs.append(extract_batch(batch[offset:offset + sub_batch_size]))
            return np.concatenate(outputs, axis=0).astype(np.float32)

        tracklets_by_camera = load_tracklets_by_camera(stage1_dir)
        tracklet_lookup = {}
        for camera_id, tracklets in tracklets_by_camera.items():
            for tracklet in tracklets:
                tracklet_lookup[(camera_id, tracklet.track_id)] = tracklet

        crop_extractor = CropExtractor(
            min_area=stage_cfg.crop.min_area,
            padding_ratio=stage_cfg.crop.padding_ratio,
            samples_per_tracklet=stage_cfg.crop.samples_per_tracklet,
            min_quality=stage_cfg.crop.get("min_quality", 0.05),
            laplacian_min_var=stage_cfg.crop.get("laplacian_min_var", 0.0),
        )

        needed_ids_by_camera = {}
        for entry in index_map:
            tracklet = tracklet_lookup.get((entry["camera_id"], entry["track_id"]))
            if tracklet is None:
                continue
            needed = needed_ids_by_camera.setdefault(entry["camera_id"], set())
            for tracklet_frame in tracklet.frames:
                needed.add(tracklet_frame.frame_id)

        frames_cache = {}
        for camera_id, needed_ids in sorted(needed_ids_by_camera.items()):
            frames_cache[camera_id] = _load_frames_for_camera(stage0_dir, camera_id, needed_ids)
            print(f"  {camera_id}: loaded {len(frames_cache[camera_id])}/{len(needed_ids)} frames")

        raw_secondary = np.zeros((len(index_map), 768), dtype=np.float32)
        valid_mask = np.zeros(len(index_map), dtype=bool)
        ordered_camera_ids = []
        missing_tracklets = 0
        missing_crops = 0

        for row_idx, entry in enumerate(index_map):
            camera_id = entry["camera_id"]
            track_id = entry["track_id"]
            ordered_camera_ids.append(camera_id)
            tracklet = tracklet_lookup.get((camera_id, track_id))
            if tracklet is None:
                missing_tracklets += 1
                continue

            scored_crops = crop_extractor.extract_crops_from_frames(
                tracklet,
                frames_cache.get(camera_id, {}),
            )
            if not scored_crops:
                missing_crops += 1
                continue

            crop_images = [crop.image for crop in scored_crops]
            crop_embeddings = extract_crops_embeddings(crop_images, batch_size=cnn_batch_size)
            if crop_embeddings.size == 0:
                missing_crops += 1
                continue

            qualities = np.array([crop.quality for crop in scored_crops], dtype=np.float32)
            logits = qualities * quality_temperature
            logits = logits - float(logits.max())
            weights = np.exp(logits)
            weights_sum = float(weights.sum())
            if weights_sum <= 0:
                missing_crops += 1
                continue
            weights = weights / weights_sum

            raw_secondary[row_idx] = (crop_embeddings * weights[:, None]).sum(axis=0).astype(np.float32)
            valid_mask[row_idx] = True

        valid_count = int(valid_mask.sum())
        if valid_count == 0:
            raise RuntimeError("CNN extraction produced zero valid embeddings")

        secondary_valid = raw_secondary[valid_mask]
        valid_camera_ids = [ordered_camera_ids[idx] for idx, is_valid in enumerate(valid_mask) if is_valid]
        print(
            f"CNN secondary raw matrix: {secondary_valid.shape} | "
            f"missing_tracklets={missing_tracklets} | missing_crops={missing_crops}"
        )

        if stage_cfg.get("camera_bn", {}).get("enabled", True):
            secondary_valid = camera_aware_batch_normalize(secondary_valid, valid_camera_ids)
            print("✓ Camera-BN applied to CNN secondary embeddings")
        else:
            print("✓ Camera-BN skipped (matches saved run config)")

        cnn_pca = PCAWhitener(n_components=target_dim)
        secondary_valid = cnn_pca.fit_transform(secondary_valid)
        pca_path = stage2_dir / "pca_transform_cnn_secondary.pkl"
        cnn_pca.save(pca_path)
        if secondary_valid.shape[1] < target_dim:
            pad_width = target_dim - secondary_valid.shape[1]
            secondary_valid = np.concatenate(
                [secondary_valid, np.zeros((secondary_valid.shape[0], pad_width), dtype=np.float32)],
                axis=1,
            )
            print(f"⚠ PCA produced {target_dim - pad_width} dims; zero-padded to {target_dim}D")
        print(f"✓ PCA whitening applied: 768D -> {secondary_valid.shape[1]}D")

        secondary_valid = l2_normalize(secondary_valid)
        secondary_full = np.zeros((len(index_map), secondary_valid.shape[1]), dtype=np.float32)
        secondary_full[valid_mask] = secondary_valid.astype(np.float32)

        secondary_path = stage2_dir / "embeddings_secondary.npy"
        np.save(secondary_path, secondary_full.astype(np.float32))
        print(f"✓ Saved CNN secondary embeddings: {secondary_path}")
        print(f"  Shape: {secondary_full.shape}")
        print(f"  Matches primary rows: {secondary_full.shape[0] == primary_embeddings.shape[0]}")
        print(f"  Zero-fill rows: {(~valid_mask).sum()}")

        del cnn_model, secondary_valid, secondary_full, raw_secondary
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        elapsed = _cnn_time.time() - _cnn_t0
        print(f"
✓ CNN secondary extraction done in {elapsed / 60:.1f} min")

## 5. Save Checkpoint

Saves stage1 + stage2 outputs + GT annotations to `/kaggle/working/checkpoint.tar.gz`.
If `embeddings_secondary.npy` exists in `stage2`, it is included automatically.
This file becomes the input for **10b**.
Stage0 frame images (~9.6 GB) are **not** included -- downstream stages do not need them.

In [ ]:
    import re as _re2

    CAM_RE2 = _re2.compile(r"^S\d{2}_c\d{3}$")
    checkpoint_path = Path("/kaggle/working/checkpoint.tar.gz")
    metadata_path   = Path("/kaggle/working/run_metadata.json")
    secondary_embeddings_path = RUN_DIR / "stage2" / "embeddings_secondary.npy"

    with open(metadata_path, "w") as f:
        json.dump({"run_name": RUN_NAME}, f)

    print(f"Building checkpoint for run: {RUN_NAME}")
    if secondary_embeddings_path.exists():
        print(f"  ✓ Secondary embeddings detected: {secondary_embeddings_path}")
    else:
        print("  ⚠ Secondary embeddings not found; checkpoint will contain primary stage2 artifacts only")
    with tarfile.open(str(checkpoint_path), "w:gz") as tar:
        tar.add(str(metadata_path), arcname="run_metadata.json")

        manifest = RUN_DIR / "stage0" / "frames_manifest.json"
        if manifest.exists():
            tar.add(str(manifest), arcname=f"{RUN_NAME}/stage0/frames_manifest.json")
            print("  + stage0/frames_manifest.json")

        for stage in ["stage1", "stage2"]:
            stage_dir = RUN_DIR / stage
            if stage_dir.exists():
                n = 0
                for fpath in stage_dir.rglob("*"):
                    if fpath.is_file():
                        tar.add(str(fpath), arcname=f"{RUN_NAME}/{stage}/{fpath.relative_to(stage_dir)}")
                        n += 1
                print(f"  + {stage}/ ({n} files)")

        # GT annotation txt files needed by stage5 eval (small text files, not videos)
        gt_count = 0
        for cam_dir in DATA_RAW.iterdir():
            if cam_dir.is_dir() and CAM_RE2.match(cam_dir.name):
                gt_file = cam_dir / "gt" / "gt.txt"
                if gt_file.exists():
                    tar.add(str(gt_file), arcname=f"gt_annotations/{cam_dir.name}/gt/gt.txt")
                    gt_count += 1
        print(f"  + gt_annotations/ ({gt_count} GT files)")

    size_mb = checkpoint_path.stat().st_size / 1024**2
    print(f"
✓ Checkpoint: {checkpoint_path}  ({size_mb:.1f} MB)")
    print("  Next: attach this notebook's output to 10b, then push 10b.")